In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import sys
sys.path.insert(0, "/Users/jihaein/Desktop/fds_main/investment-evidence-system")

In [2]:
%load_ext autoreload
%autoreload 2
from backtest.engine import load_daily_price, run_strategy
import numpy as np

In [3]:
grid = load_daily_price()
print(grid.shape)            # expect (1249, 120)

(1249, 120)


/Users/jihaein/Desktop/fds_main/investment-evidence-system/backtest/engine.py:35: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return close.pct_change() * 100   # daily % return (same units as the old FLUC_RT)


In [4]:
# quick random-20 baseline sanity check
rng = np.random.default_rng(42)
basket = grid[list(rng.choice(grid.columns, 20, replace=False))]
print("무손절 final W:", run_strategy(basket, {}).iloc[-1])

무손절 final W: 12528378.179230237


In [5]:
window_end = grid.index[grid.index >= "2016-01-04"][-1]
for s in grid.columns:
    last = grid[s].last_valid_index()
    if last is not None and last < window_end:
        print(s, "delists", last)

In [6]:
line = run_strategy(grid, {})       # 무손절 on the full 19 stocks
print("any NaN?", line.isna().any())   # expect False
print("final W:", line.iloc[-1])       # expect a real number

any NaN? False
final W: 14242245.391724978


In [7]:
hi = run_strategy(grid, {}, recovery_rate=1.0).iloc[-1]   # exit at last price
lo = run_strategy(grid, {}, recovery_rate=0.0).iloc[-1]   # total loss on delisting
print("recovery 1.0:", hi)
print("recovery 0.0:", lo)
print("gap:", hi - lo)

recovery 1.0: 14242245.391724978
recovery 0.0: 14242245.391724978
gap: 0.0
